# Robustness tests for the NFFT/LSP return periods

Reproduces the robustness analyses of the study (table numbers refer to the accompanying article):

1. checks that the estimator code below reproduces Tables 5 and 6 exactly;
2. split-sample evaluation on the clusters scored by every method, with a Gutenberg–Richter (GR) baseline and a control forecast (0.5 × mean) — **Table 9**;
3. rate error |log10(T/T_obs)| — **Table 10**;
4. permutation tests of the spectral peaks, Benjamini–Hochberg FDR and LSP Baluev FAP — **Table 11**;
5. synthetic recovery experiments — **Table 12**;
6. resolution uncertainty ΔT = T²/D and run time.

The pipeline functions (`recount_day_number`, `remove_aftershocks`, `split_clusters_80_20`) are taken unchanged from `code/4_statistics_NFFT_LSP.py`; the input is `data/clustering/`.
Environment used: Python 3.10, numpy 2.2.6, pandas 2.3.3, scipy 1.15.3, astropy 6.1.7, nfft 0.1.

In [1]:
# ---- parameters ----
import os
REPO = os.environ.get('QRP_REPO', os.path.abspath('..'))   # repository root (this notebook lives in code/)
B = 999                               # permutations per peak
RECOMPUTE_PERMUTATIONS = False        # True: rerun the ~12-minute permutation test; False: load permutation_B999.csv
OUT = os.path.join(REPO, 'results', 'robustness')   # inputs (precomputed nulls) and outputs

In [2]:
import os, sys, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
src=open(f'{REPO}/code/4_statistics_NFFT_LSP.py').read()
cut=src.index('# use_split = split_data')
NS={}
import io, contextlib
exec(compile(src[:cut],'pipeline_defs','exec'),NS)
recount_day_number=NS['recount_day_number']; remove_aftershocks=NS['remove_aftershocks']
split_clusters_80_20=NS['split_clusters_80_20']
from nfft import nfft_adjoint
from scipy.signal import find_peaks
from astropy.timeseries import LombScargle
COUNTRIES=[l.strip() for l in open(f'{REPO}/code/countries.txt') if l.strip()]
quiet=lambda: contextlib.redirect_stdout(io.StringIO())

def stats_series(country, M):
    """Replicates the statistics loop: returns {mapped_id: analysis_data} for clusters with >=2 declustered events,
    plus mapping. analysis_data = declustered M>=M series (this is what data4calculation holds)."""
    d=pd.read_csv(f'{REPO}/data/clustering/clustered_earthquakes_{country}.csv')
    d.columns=d.columns.str.lower(); d=d.loc[:,~d.columns.duplicated()]
    d['time']=pd.to_datetime(d['time'],format='mixed',errors='coerce')
    keys=[]; out={}
    for cid in sorted(d['cluster'].unique()):
        if cid==-1: continue
        keys.append(cid)
        cd=d[d['cluster']==cid].copy()
        if len(cd)<10: continue
        f=cd[cd['mag']>=M].copy()
        if len(f)<2: continue
        with quiet():
            a=recount_day_number(f); a=remove_aftershocks(a,180); a=recount_day_number(a)
        if len(a)<2: continue
        out[cid]=a
    mapping={old:i+1 for i,old in enumerate(sorted(keys))}
    return {mapping[k]:v for k,v in out.items()}

def spectral_input(a):
    """What run_NFFT / run_LSP see in non-split mode: data4calculation cluster -> recount -> decluster -> recount."""
    if len(a)<10: return None
    a=a.copy()
    with quiet():
        a=recount_day_number(a); a=remove_aftershocks(a,180); a=recount_day_number(a)
    if len(a)<5: return None
    a=a.sort_values('Day_Number',kind='mergesort').reset_index(drop=True)
    return a['Day_Number'].to_numpy(dtype=float), a['mag'].to_numpy(dtype=float)

def band(day):
    num_day=day.max()-day.min()
    if num_day==0: num_day=1
    g=np.diff(np.sort(day)); g=g[g>0]
    pmin=2.0*np.median(g) if g.size else 1.0
    pmax=num_day/3.0
    return num_day, num_day/(0.4999*pmax), num_day/(0.4999*pmin)

def nfft_est(day, mag, return_amp=False):
    N=len(mag); num_day,lb,ub=band(day)
    x=day-day.min(); xn=x*0.4999/x.max() if x.max()>0 else np.zeros_like(x)
    yc=mag-mag.mean()
    NF=max(2*(int(np.ceil(ub))+1), 2*(N//2))
    k=np.arange(-NF//2,NF//2)
    spec=np.abs(nfft_adjoint(xn.astype(float),yc,NF,sigma=5))
    pos=k>0; xf=k[pos].astype(float); amp=spec[pos]/N
    pk,_=find_peaks(amp,distance=3,prominence=0.01,height=0.01)
    best=0;bf=0
    for p in pk:
        if lb<=xf[p]<=ub and amp[p]>best: best=amp[p]; bf=xf[p]
    if best<=0: return (None,0.0) if return_amp else None
    T=(num_day/(bf*0.4999))/365.25
    return (T,best) if return_amp else T

def lsp_est(day, mag, return_power=False):
    N=len(mag); x=day-day[0]; num_day=day.max()-day.min()
    xn=x*0.4999/x.max()
    f,p=LombScargle(xn,mag).autopower(minimum_frequency=0.01,maximum_frequency=N,samples_per_peak=10)
    g=np.diff(np.sort(x.astype(float))); g=g[g>0]
    pmin=2.0*np.median(g) if g.size else 1.0; pmax=num_day/3.0
    lo=num_day/(0.4999*pmax); hi=num_day/(0.4999*pmin)
    m=(f>=lo)&(f<=hi)
    if not m.any(): return (None,0.0) if return_power else None
    i=np.argmax(p[m]); bf=f[m][i]
    T=(num_day/bf/0.4999)/365.25
    return (T,p[m][i]) if return_power else T


In [3]:
import pickle, time
t0=time.perf_counter()
SERIES={(c,M):stats_series(c,M) for c in COUNTRIES for M in range(4,9)}
print(f'declustered series prepared in {time.perf_counter()-t0:.1f} s')

declustered series prepared in 32.5 s


## 1. The estimators reproduce Tables 5 and 6

In [4]:
R=f'{REPO}/results'; rows=[]
for M in range(4,9):
    nf=pd.read_csv(f'{R}/NFFT_M{M}.csv'); ls=pd.read_csv(f'{R}/LSP-M{M}.csv')
    for c in COUNTRIES:
        for cid,a in SERIES[(c,M)].items():
            inp=spectral_input(a); Tn=Tl=None
            if inp is not None: Tn=nfft_est(*inp); Tl=lsp_est(*inp)
            rn=pd.to_numeric(nf[(nf.country==c)&(nf.cluster==cid)]['return_P_nomes'],errors='coerce')
            rl=pd.to_numeric(ls[(ls.country==c)&(ls.cluster==cid)]['return_p_years'],errors='coerce')
            rows.append((M,c,cid,Tn,rn.iloc[0] if len(rn) else np.nan,Tl,rl.iloc[0] if len(rl) else np.nan))
V=pd.DataFrame(rows,columns='M country cl Tn refn Tl refl'.split())
for a,b in (('Tn','refn'),('Tl','refl')):
    x=pd.to_numeric(V[a],errors='coerce'); both=x.notna()&V[b].notna()
    print(a,'cells:',both.sum(),'identical to 3 dp:',(x[both].round(3)==V[b][both].round(3)).sum(),
          '| only here:',(x.notna()&V[b].isna()).sum(),'| only in results:',(x.isna()&V[b].notna()).sum())

Tn cells: 181 identical to 3 dp: 181 | only here: 0 | only in results: 0
Tl cells: 183 identical to 3 dp: 183 | only here: 0 | only in results: 0


## 2–3. Split-sample evaluation (Table 9) and rate error (Table 10)

In [5]:
import pickle, numpy as np, pandas as pd
from datetime import timedelta
from scipy import stats as sst

def predict_errors(train, test, T_days):
    """Exactly the stepping protocol of predict_and_error(_nfftlsp): returns list of error_days."""
    tt=pd.to_datetime(test['time']).dt.date.sort_values().to_numpy()
    if len(tt)==0 or T_days is None or int(T_days)<=0: return []
    last=pd.to_datetime(train['time']).max().date()
    step=int(T_days); cur=last+timedelta(days=step); errs=[]
    while cur<=tt.max():
        e=np.abs(np.array([(t-cur).days for t in tt])); errs.append(int(e.min()))
        cur+=timedelta(days=step)
    return errs

def splits_stats(a):
    with quiet(): tr,te=split_clusters_80_20(a.copy())
    return tr,te

def splits_spec(a):
    if len(a)<10: return None
    with quiet():
        tr,te=split_clusters_80_20(a.copy())
        if len(tr)<5: return None
        tr=recount_day_number(tr); tr=remove_aftershocks(tr,180); tr=recount_day_number(tr)
        if len(tr)<5: return None
        te=recount_day_number(te); te=remove_aftershocks(te,180); te=recount_day_number(te)
    return tr,te

def train_gaps(tr):
    t=pd.to_datetime(tr['time'])
    return np.array([(t.iloc[i].date()-t.iloc[i-1].date()).days for i in range(1,len(t))])

def run(extra=None):
    """extra: dict name -> f(train_df, M, country, cid) -> T_days, evaluated on the statistics split."""
    out={}  # (method,M) -> list of (country,cid,errs,T_days,train,test)
    for M in range(4,9):
        for c in COUNTRIES:
            S=SERIES[(c,M)]
            for cid,a in S.items():
                g=train_gaps(a) if len(a)>1 else np.array([])
                if len(g)==0 or g.mean()<=0: pass
                else:
                    tr,te=splits_stats(a)
                    if len(tr) and len(te):
                        gg=train_gaps(tr)
                        Ts={'mean':np.mean(gg),'mode':sst.mode(gg,keepdims=True).mode[0],'median':np.median(gg)}
                        if extra:
                            for k,f in extra.items(): Ts[k]=f(tr,M,c,cid)
                        for k,T in Ts.items():
                            out.setdefault((k,M),[]).append((c,cid,predict_errors(tr,te,T),T,tr,te))
                sp=splits_spec(a)
                if sp:
                    tr,te=sp
                    d=tr.sort_values('Day_Number',kind='mergesort')
                    day=d['Day_Number'].to_numpy(float); mag=d['mag'].to_numpy(float)
                    Tn=nfft_est(day,mag)
                    if Tn: 
                        gd=max(1,int(round(Tn*365.25)))
                        out.setdefault(('NFFT',M),[]).append((c,cid,predict_errors(tr,te,gd),gd,tr,te))
                    Tl=lsp_est(day,mag)
                    if Tl:
                        pdays=Tl*365.25
                        out.setdefault(('LSP',M),[]).append((c,cid,predict_errors(tr,te,pdays),pdays,tr,te))
    return out

def metrics(out, Y=365.0):
    rows=[]
    for (k,M),L in out.items():
        e=np.concatenate([np.array(x[2],float) for x in L]) if L else np.array([])
        if e.size==0: rows.append((k,M,0,np.nan,np.nan,np.nan)); continue
        rows.append((k,M,e.size,np.mean(e)/Y,np.sqrt(np.mean(e**2))/Y,np.median(e)/Y))
    return pd.DataFrame(rows,columns=['method','M','n','MAE','RMSE','MedAE'])


In [6]:
RAW={}
def raw(c):
    if c not in RAW:
        d=pd.read_csv(f'{REPO}/data/clustering/clustered_earthquakes_{c}.csv'); d.columns=d.columns.str.lower(); d=d.loc[:,~d.columns.duplicated()]
        d['time']=pd.to_datetime(d['time'],format='mixed',errors='coerce',utc=True)
        keys=sorted(k for k in d['cluster'].unique() if k!=-1); mp={o:i+1 for i,o in enumerate(keys)}
        d['cl']=d['cluster'].map(mp); RAW[c]=d
    return RAW[c]
MC=4.5; T0=pd.Timestamp('1973-01-01',tz='UTC')
def gr_T(tr,M,c,cid):
    d=raw(c); cut=pd.to_datetime(tr['time'],utc=True).max()
    s=d[(d.cl==cid)&(d.time>=T0)&(d.time<=cut)&(d.mag>=MC)]
    yrs=(cut-T0).days/365.25
    if len(s)<10:   # fallback: whole record of the cluster up to the cut-off
        s=d[(d.cl==cid)&(d.time<=cut)&(d.mag>=MC)]
        yrs=(cut-d[d.cl==cid].time.min()).days/365.25
    if len(s)<5 or yrs<=0: return None
    b=np.log10(np.e)/(s.mag.mean()-(MC-0.005))
    lam=len(s)/yrs*10**(-b*(M-MC))
    return 365.25/lam

extra={'GR':gr_T,'half_mean':lambda tr,M,c,cid: 0.5*np.mean(train_gaps(tr))}
t0=time.perf_counter(); o=run(extra); print(f'split-sample evaluation in {time.perf_counter()-t0:.1f} s')
for k in list(o):
    if k[0]=='GR': o[k]=[x for x in o[k] if x[3] is not None]

split-sample evaluation in 14.9 s


In [7]:
def tobs(tr,te):
    a=pd.to_datetime(tr['time']).max(); b=pd.to_datetime(te['time']).max(); n=len(te)
    return ((b-a).days/365.25)/n if n>0 else np.nan
meth=['mean','mode','median','NFFT','LSP','GR','half_mean']
T9={};T10={};NCL={}
for M in range(4,8):
    # clusters on which all five original methods make at least one forecast
    ks=lambda k:{(x[0],x[1]) for x in o.get((k,M),[]) if len(x[2])>0}
    common=set.intersection(*[ks(k) for k in ['mean','mode','median','NFFT','LSP']]); NCL[M]=len(common)
    ref={(x[0],x[1]):tobs(x[4],x[5]) for x in o[('mean',M)] if (x[0],x[1]) in common}
    for k in meth:
        L=[x for x in o[(k,M)] if (x[0],x[1]) in common]
        e=np.concatenate([np.array(x[2],float) for x in L])/365.0
        T9[(k,M)]=dict(MAE=e.mean(),RMSE=np.sqrt((e**2).mean()),MedAE=np.median(e),forecasts=e.size)
        s=np.array([np.log10((x[3]/365.25)/ref[(x[0],x[1])]) for x in L])
        T10[(k,M)]=dict(median_abs_log10=np.median(np.abs(s)),within_factor2_pct=100*np.mean(np.abs(s)<=np.log10(2)),
                         overestimate_pct=100*np.mean(s>0),median_signed_log10=np.median(s))
t9=pd.DataFrame(T9).T; t9.index.names=['method','M']; t10=pd.DataFrame(T10).T; t10.index.names=['method','M']
print('clusters scored:',NCL)
for v in ['MAE','RMSE','MedAE','forecasts']: print(v); print(t9[v].unstack().loc[meth].round(3))
for v in ['median_abs_log10','within_factor2_pct','overestimate_pct']: print(v); print(t10[v].unstack().loc[meth].round(3))
t9.to_csv(f'{OUT}/table9_common_clusters.csv'); t10.to_csv(f'{OUT}/table10_rate_error.csv')

clusters scored: {4: 48, 5: 48, 6: 28, 7: 10}
MAE
M              4      5      6      7
method                               
mean       0.248  0.402  0.801  1.466
mode       0.288  0.635  1.406  1.832
median     0.265  0.468  0.871  1.385
NFFT       0.299  0.478  0.988  1.291
LSP        0.261  0.406  0.976  1.551
GR         0.240  0.243  0.410  1.317
half_mean  0.253  0.402  0.803  1.416
RMSE
M              4      5      6      7
method                               
mean       0.317  0.838  1.654  2.396
mode       0.423  1.345  2.598  2.977
median     0.364  1.000  1.759  2.358
NFFT       0.395  0.898  2.067  2.274
LSP        0.345  0.789  2.001  2.580
GR         0.295  0.342  0.892  2.100
half_mean  0.338  0.832  1.657  2.400
MedAE
M              4      5      6      7
method                               
mean       0.216  0.238  0.326  0.918
mode       0.227  0.277  0.477  0.899
median     0.216  0.248  0.334  0.703
NFFT       0.249  0.260  0.288  0.636
LSP        0.227  0.237  0.

In [8]:
# cluster bootstrap of the MAE difference (4,000 resamples of the common clusters)
rng=np.random.default_rng(1); BOOT=[]
for M in range(4,8):
    ks=lambda k:{(x[0],x[1]) for x in o.get((k,M),[]) if len(x[2])>0}
    common=sorted(set.intersection(*[ks(k) for k in ['mean','mode','median','NFFT','LSP']]))
    E={k:{(x[0],x[1]):np.array(x[2],float)/365 for x in o[(k,M)] if (x[0],x[1]) in common} for k in ['NFFT','LSP','mean','median']}
    for a,b in (('NFFT','mean'),('NFFT','median'),('LSP','mean')):
        dd=[]
        for _ in range(4000):
            cs=[common[i] for i in rng.integers(0,len(common),len(common))]
            dd.append(np.concatenate([E[a].get(c,np.array([])) for c in cs]).mean()-np.concatenate([E[b].get(c,np.array([])) for c in cs]).mean())
        lo,hi=np.percentile(dd,[2.5,97.5]); BOOT.append((M,a,b,lo,hi))
BOOT=pd.DataFrame(BOOT,columns=['M','a','b','ci_low','ci_high']); print(BOOT.round(3)); BOOT.to_csv(f'{OUT}/bootstrap_mae_difference.csv',index=False)

    M     a       b  ci_low  ci_high
0   4  NFFT    mean   0.010    0.111
1   4  NFFT  median   0.004    0.072
2   4   LSP    mean  -0.017    0.048
3   5  NFFT    mean  -0.005    0.187
4   5  NFFT  median  -0.063    0.088
5   5   LSP    mean  -0.067    0.090
6   6  NFFT    mean   0.016    0.437
7   6  NFFT  median  -0.055    0.373
8   6   LSP    mean   0.013    0.461
9   7  NFFT    mean  -0.313    0.023
10  7  NFFT  median  -0.238    0.062
11  7   LSP    mean  -0.232    0.438


In [9]:
# the same metrics when every method is scored on every cluster it can handle (the earlier, unequal comparison)
full=metrics(o); print(full.pivot(index='method',columns='M',values='MAE').round(3))

M              4      5      6      7       8
method                                       
GR         0.240  0.244  0.439  1.727  10.512
LSP        0.291  0.426  0.976  1.561     NaN
NFFT       0.300  0.460  0.928  1.291     NaN
half_mean  0.268  0.417  0.838  2.945  15.852
mean       0.265  0.415  0.841  2.872  16.083
median     0.275  0.460  0.895  2.673  14.899
mode       0.322  0.670  1.444  4.202  17.488


## 4. Permutation tests (Table 11)

In [10]:
def permutation_part(M, start, stop, B):
    MS=[M]; CS=COUNTRIES[start:stop]; rows=[]
    rng=np.random.default_rng([20261027,M,start])
    def nstat(day,mag):
        r=nfft_est(day,mag,return_amp=True); return r
    def lstat(day,mag):
        r=lsp_est(day,mag,return_power=True); return r
    rows=[]
    for M in MS:
        for c in CS:
            S=SERIES[(c,M)]
            for cid,a in S.items():
                inp=spectral_input(a)
                if inp is None: continue
                day,mag=inp; span_y=(day.max()-day.min())/365.25
                Tn,An=nstat(day,mag); Tl,Pl=lstat(day,mag)
                if Tn is None and Tl is None: continue
                # analytic FAP for LSP over the search band
                fap=np.nan
                if Tl is not None:
                    x=day-day[0]; xn=x*0.4999/x.max(); num_day=day.max()-day.min()
                    g=np.diff(np.sort(x)); g=g[g>0]; lo=num_day/(0.4999*(num_day/3)); hi=num_day/(0.4999*(2*np.median(g)))
                    ls=LombScargle(xn,mag)
                    fap=float(ls.false_alarm_probability(Pl,minimum_frequency=lo,maximum_frequency=hi,method='baluev'))
                nA=[];nP=[];nTn=[];nTl=[]
                for b in range(B):
                    m=rng.permutation(mag)
                    t,aa=nstat(day,m); nA.append(aa); nTn.append(t)
                    t2,pp=lstat(day,m); nP.append(pp); nTl.append(t2)
                nA=np.array(nA); nP=np.array(nP)
                pn=(1+(nA>=An).sum())/(B+1) if Tn is not None else np.nan
                pl=(1+(nP>=Pl).sum())/(B+1) if Tl is not None else np.nan
                rows.append(dict(M=M,country=c,cl=cid,N=len(mag),span=span_y,Tn=Tn,Tl=Tl,p_nfft=pn,p_lsp=pl,fap_lsp=fap,
                                 nullTn_med=np.nanmedian([t for t in nTn if t] or [np.nan]),nullTl_med=np.nanmedian([t for t in nTl if t] or [np.nan])))
    return rows
if RECOMPUTE_PERMUTATIONS:
    rows=[]
    for M in range(4,9):
        for s in (0,7,14,21): rows+=permutation_part(M,s,s+7,B)
    PERM=pd.DataFrame(rows); PERM.to_csv(f'{OUT}/permutation_B{B}.csv',index=False)
else:
    PERM=pd.read_csv(f'{OUT}/permutation_B{B}.csv')
def bh(p):
    p=np.asarray(p,float); m=~np.isnan(p); q=np.full_like(p,np.nan); pp=p[m]; k=len(pp)
    i=np.argsort(pp); r=pp[i]*k/np.arange(1,k+1); r=np.minimum.accumulate(r[::-1])[::-1]; qq=np.empty(k); qq[i]=np.minimum(r,1); q[m]=qq; return q
PERM['q_nfft']=bh(PERM.p_nfft); PERM['q_lsp']=bh(PERM.p_lsp)
t11=PERM.groupby('M').agg(NFFT_estimates=('Tn',lambda s:s.notna().sum()),NFFT_p05=('p_nfft',lambda s:(s<0.05).sum()),NFFT_q05=('q_nfft',lambda s:(s<0.05).sum()),
    LSP_estimates=('Tl',lambda s:s.notna().sum()),LSP_p05=('p_lsp',lambda s:(s<0.05).sum()),LSP_q05=('q_lsp',lambda s:(s<0.05).sum()),LSP_FAP05=('fap_lsp',lambda s:(s<0.05).sum()))
print(t11); print(t11.sum()); t11.to_csv(f'{OUT}/table11_permutation_summary.csv')
print(PERM[PERM.M==7][['country','cl','N','Tn','p_nfft','Tl','p_lsp','fap_lsp']].round(3))

   NFFT_estimates  NFFT_p05  NFFT_q05  LSP_estimates  LSP_p05  LSP_q05  \
M                                                                        
4              67         2         0             67        4        0   
5              60         2         0             60        3        0   
6              39         3         0             40        1        0   
7              13         1         0             14        0        0   
8               2         0         0              2        0        0   

   LSP_FAP05  
M             
4          2  
5          2  
6          0  
7          0  
8          0  
NFFT_estimates    181
NFFT_p05            8
NFFT_q05            0
LSP_estimates     183
LSP_p05             8
LSP_q05             0
LSP_FAP05           4
dtype: int64
              country  cl   N      Tn  p_nfft      Tl  p_lsp  fap_lsp
0         New Zealand   1  35   9.812   0.862   7.519  0.557    0.707
1         New Zealand   2  11     NaN     NaN  16.395  0.651    0.875

## 5. Synthetic recovery (Table 12)

In [11]:
import pickle,numpy as np,pandas as pd
rng=np.random.default_rng(7)
# (a) magnitude modulation on real M4+ sampling times
rows=[]
for c in COUNTRIES:
    S=SERIES[(c,4)]
    for cid,a in S.items():
        inp=spectral_input(a)
        if inp is None: continue
        day,mag=inp; span=day.max()-day.min(); g=np.diff(day); pmin=2*np.median(g[g>0])
        sig=mag.std()
        for k0 in (4,8,16):
            T0=span/k0
            if T0<pmin*1.2: continue
            for snr in (0.5,1.0):
                for rep in range(5):
                    y=mag.mean()+snr*sig*np.sin(2*np.pi*day/T0+rng.uniform(0,2*np.pi))+rng.normal(0,sig,len(day))
                    Tn=nfft_est(day,y); Tl=lsp_est(day,y)
                    dT=(T0/365.25)**2/(span/365.25)
                    rows.append(dict(kind='mag-mod',c=c,cl=cid,N=len(day),T0=T0/365.25,snr=snr,Tn=Tn,Tl=Tl,dT=dT))
A=pd.DataFrame(rows)
def ok(T,T0,dT): return np.where(pd.isna(T),False,np.abs(T.astype(float)-T0)<=np.maximum(dT,0.05*T0))
A['okn']=ok(A.Tn,A.T0,A.dT); A['okl']=ok(A.Tl,A.T0,A.dT)
print('(a) magnitude-modulated signal on real sampling times: recovery within dT')
print(A.groupby('snr')[['okn','okl']].mean().round(3), len(A))
A['relN']=np.abs(A.Tn.astype(float)/A.T0-1); A['relL']=np.abs(A.Tl.astype(float)/A.T0-1)
print(A.groupby('snr')[['relN','relL']].median().round(3))
# (b) quasi-periodic occurrence with iid magnitudes; (c) Poisson occurrence
allm=np.concatenate([SERIES[(c,6)][k]['mag'].to_numpy() for c in COUNTRIES for k in SERIES[(c,6)]])
rows=[]
SPAN=126*365.25
for kind in ('renewal','poisson'):
    for T0y in (2,5,10):
        for rep in range(100):
            if kind=='renewal':
                t=np.cumsum(rng.gamma(25,T0y*365.25/25,size=int(SPAN/(T0y*365.25))*2+10))  # CV=0.2
            else:
                t=np.cumsum(rng.exponential(T0y*365.25,size=int(SPAN/(T0y*365.25))*3+10))
            t=np.floor(t[t<SPAN]); t=np.unique(t)
            if len(t)<10: continue
            y=rng.choice(allm,len(t))
            rows.append(dict(kind=kind,T0=T0y,N=len(t),Tn=nfft_est(t,y),Tl=lsp_est(t,y),obs_mean=np.mean(np.diff(t))/365.25))
Bq=pd.DataFrame(rows)
for col in ('Tn','Tl'): Bq[col]=Bq[col].astype(float)
Bq['okn']=np.abs(Bq.Tn-Bq.T0)<=0.1*Bq.T0; Bq['okl']=np.abs(Bq.Tl-Bq.T0)<=0.1*Bq.T0
print('\n(b,c) occurrence-periodic vs Poisson, iid magnitudes, 126-yr record')
print(Bq.groupby(['kind','T0']).agg(N=('N','median'),Tn_med=('Tn','median'),Tn_iqr=('Tn',lambda s:s.quantile(.75)-s.quantile(.25)),Tl_med=('Tl','median'),okn=('okn','mean'),okl=('okl','mean'),resolved_n=('Tn',lambda s:s.notna().mean())).round(2))
A.to_csv(f'{OUT}/synthetic_magnitude_modulation.csv',index=False); Bq.to_csv(f'{OUT}/synthetic_occurrence.csv',index=False)


(a) magnitude-modulated signal on real sampling times: recovery within dT
       okn    okl
snr              
0.5  0.559  0.615
1.0  0.936  0.931 1960
      relN   relL
snr              
0.5  0.059  0.048
1.0  0.000  0.012



(b,c) occurrence-periodic vs Poisson, iid magnitudes, 126-yr record
               N  Tn_med  Tn_iqr  Tl_med   okn   okl  resolved_n
kind    T0                                                      
poisson 2   63.0    5.89    6.54    6.56  0.00  0.00        1.00
        5   24.0   11.48    8.15   16.16  0.05  0.01        1.00
        10  14.0   18.90    9.37   21.48  0.07  0.02        0.99
renewal 2   63.0    7.23    7.78    6.98  0.00  0.00        1.00
        5   25.0   16.03    9.39   16.41  0.00  0.00        1.00
        10  12.0   24.34    5.54   24.53  0.00  0.00        0.86


## 6. Resolution uncertainty and run time

In [12]:
import pickle,time,numpy as np,pandas as pd
rows=[];tn=[];tl=[];Ns=[]
for M in range(4,9):
    for c in COUNTRIES:
        S=SERIES[(c,M)]
        for cid,a in S.items():
            inp=spectral_input(a)
            if inp is None: continue
            day,mag=inp; span=(day.max()-day.min())/365.25
            t0=time.perf_counter(); Tn=nfft_est(day,mag); t1=time.perf_counter(); Tl=lsp_est(day,mag); t2=time.perf_counter()
            tn.append(t1-t0); tl.append(t2-t1); Ns.append(len(day))
            for meth,T in (('NFFT',Tn),('LSP',Tl)):
                if T: rows.append(dict(M=M,c=c,cl=cid,meth=meth,T=T,span=span,dT=T*T/span,N=len(day)))
d=pd.DataFrame(rows); d['rel']=d.dT/d['T']
print(d.groupby(['meth','M']).agg(n=('T','size'),med_T=('T','median'),med_dT=('dT','median'),med_rel=('rel','median'),max_rel=('rel','max')).round(3))
print('example Japan c1:'); print(d[(d.c=='Japan')&(d.cl==1)][['meth','M','T','dT','span','N']].round(3))
print('NFFT time per cluster ms: median %.2f max %.2f; LSP median %.2f max %.2f; N range %d-%d'%(1e3*np.median(tn),1e3*max(tn),1e3*np.median(tl),1e3*max(tl),min(Ns),max(Ns)))
print('total spectral time all 335 cells: %.2f s'%(sum(tn)+sum(tl)))
d.to_csv(f'{OUT}/period_uncertainty.csv',index=False)


         n   med_T  med_dT  med_rel  max_rel
meth M                                      
LSP  4  67   6.143   0.392    0.064    0.333
     5  60   5.774   0.414    0.065    0.333
     6  40  13.123   1.473    0.116    0.333
     7  14  10.259   0.854    0.083    0.250
     8   2  29.844   8.756    0.283    0.333
NFFT 4  67   3.723   0.134    0.035    0.250
     5  60   4.426   0.171    0.044    0.286
     6  39   7.465   0.638    0.080    0.286
     7  13   9.599   0.739    0.077    0.250
     8   2  24.711   5.842    0.236    0.250
example Japan c1:
     meth  M      T     dT     span    N
0    NFFT  4  1.803  0.026  126.160  144
1     LSP  4  1.953  0.030  126.160  144
134  NFFT  5  1.803  0.026  126.160  144
135   LSP  5  1.953  0.030  126.160  144
254  NFFT  6  1.947  0.030  125.577  141
255   LSP  6  1.950  0.030  125.577  141
333  NFFT  7  8.972  0.641  125.577   71
334   LSP  7  7.053  0.396  125.577   71
NFFT time per cluster ms: median 0.36 max 0.99; LSP median 0.45 max 1.08;

## 7. Occurrence-based estimators (Algorithm 4, Tables 13–14)

NFFT-S: Schuster spectrum of the event times via the adjoint NFFT (unit weights, 4x oversampling). LSP-C: Lomb–Scargle periodogram of event counts in bins of a quarter of the median gap. Band: 0.6 x median gap to span/3. Null: raw times resampled with 2-yr Gaussian jitter, magnitudes permuted, same 180-day declustering (`fast_decluster`, checked identical to `remove_aftershocks`).

In [13]:
RECOMPUTE_OCC_NULL = False   # True: recompute the 199-catalog null tests (~4 min serial); False: load occ_eval_B199.csv
B_OCC = 199

In [14]:
import numpy as np
from nfft import nfft_adjoint
from astropy.timeseries import LombScargle
OS=4                      # frequency oversampling
def _band(day):
    D=day.max()-day.min(); g=np.diff(np.sort(day)); g=g[g>0]
    return D, 0.6*np.median(g), D/3.0          # periods: from 0.6 x median gap to a third of the span
def schuster_nfft(day, return_all=False):
    """Schuster (Rayleigh) power of the event times, P(f)=|sum exp(2 pi i f t_j)|^2/N, evaluated with the adjoint NFFT."""
    day=np.asarray(day,float); N=len(day); D,plo,phi=_band(day)
    s=0.4999/OS; x=s*(day-day.min())/D            # nodes in [0, 0.4999/OS]
    kmax=int(np.ceil(D/(s*plo)))+2; NF=2*kmax
    k=np.arange(-NF//2,NF//2)
    P=np.abs(nfft_adjoint(x,np.ones(N),NF,sigma=5))**2/N
    pos=k>0; k=k[pos]; P=P[pos]
    per=D/(s*k)                                   # period in days for each k
    m=(per>=plo)&(per<=phi)
    if not m.any(): return (None,0.0,None) if return_all else None
    i=np.argmax(P[m]); T=per[m][i]
    return (T,P[m][i],(per[m],P[m])) if return_all else T
def lsp_counts(day, return_all=False):
    """Lomb-Scargle periodogram of event counts in bins of a quarter of the median gap."""
    day=np.asarray(day,float); D,plo,phi=_band(day); g=np.diff(np.sort(day)); g=g[g>0]
    b=max(1.0,0.25*np.median(g)); edges=np.arange(day.min(),day.max()+b,b)
    c,_=np.histogram(day,edges); t=0.5*(edges[1:]+edges[:-1])
    f=np.linspace(1/phi,1/plo,4000)
    ls=LombScargle(t,c); p=ls.power(f)
    i=np.argmax(p); T=1/f[i]
    if return_all: return T,p[i],ls.false_alarm_probability(p[i],minimum_frequency=1/phi,maximum_frequency=1/plo,method='baluev')
    return T
def null_sims(day, B=199, rng=None):
    """Declustering-consistent Poisson null: shifted-exponential gaps with the observed minimum and mean gap."""
    day=np.sort(np.asarray(day,float)); g=np.diff(day); N=len(day); D=day[-1]-day[0]
    gmin=g.min(); mu=g.mean()
    out=[]
    for _ in range(B):
        gg=gmin+rng.exponential(max(mu-gmin,1e-6),N-1)
        tt=np.concatenate([[0],np.cumsum(gg)]); tt=tt*D/tt[-1]   # same N and span
        out.append(tt)
    return out
def gated(day, B=199, rng=None, alpha=0.05, which='nfft'):
    if which=='nfft':
        T,P,_=schuster_nfft(day,True)
        if T is None: return None,1.0
        nul=[schuster_nfft(s,True)[1] for s in null_sims(day,B,rng)]
    else:
        T,P,_=lsp_counts(day,True)
        nul=[lsp_counts(s,True)[1] for s in null_sims(day,B,rng)]
    p=(1+np.sum(np.array(nul)>=P))/(B+1)
    return T,p

def fast_decluster(day, mag, W=180):
    """numpy re-implementation of remove_aftershocks() for one cluster (day sorted ascending, recounted from 0)."""
    day=np.asarray(day,int); mag=np.asarray(mag,float)
    o=np.argsort(day,kind='mergesort'); day=day[o]; mag=mag[o]
    sel=[]; start=0; mx=day.max()
    lo=0
    while True:
        end=start+W-1
        hi=np.searchsorted(day,end,side='right'); lo=np.searchsorted(day,start,side='left')
        if hi>lo: sel.append(lo+int(np.argmax(mag[lo:hi])))
        start=end+1
        if start>mx: break
    d=day[sel]; m=mag[sel]; keep=np.ones(len(d),bool)
    for i in range(1,len(d)):
        if keep[i-1] and d[i]-d[i-1]<=W:
            if m[i]<=m[i-1]: keep[i]=False
            else: keep[i-1]=False
    d=d[keep]; return d-d[0], m[keep]


In [15]:
# fast_decluster reproduces remove_aftershocks exactly
bad=n=0
for M in (4,6,7):
    for c in COUNTRIES:
        d=pd.read_csv(f'{REPO}/data/clustering/clustered_earthquakes_{c}.csv'); d.columns=d.columns.str.lower(); d=d.loc[:,~d.columns.duplicated()]
        d['time']=pd.to_datetime(d['time'],format='mixed',errors='coerce')
        for cid,a in SERIES[(c,M)].items():
            f=d[(d.cluster==cid)&(d.mag>=M)].copy()
            with quiet(): f=recount_day_number(f)
            dd,_=fast_decluster(f['Day_Number'].to_numpy(),f['mag'].to_numpy()); n+=1
            bad+=not np.array_equal(dd,a['Day_Number'].to_numpy()-a['Day_Number'].min())
print('clusters checked',n,'mismatches',bad)

clusters checked 161 mismatches 0


In [16]:
import numpy as np,pandas as pd
rng=np.random.default_rng(11); SPAN=126*365.25; rows=[]
def renewal(T0,cv,miss=0.0,extra=0.0):
    k=1/cv**2
    t=np.cumsum(rng.gamma(k,T0*365.25/k,size=int(SPAN/(T0*365.25))*2+10)); t=t[t<SPAN]
    if miss>0: t=t[rng.random(len(t))>=miss]
    if extra>0:
        ne=rng.poisson(extra*len(t)); t=np.sort(np.concatenate([t,rng.uniform(0,SPAN,ne)]))
    return np.unique(np.floor(t))
for kind,cv,miss,extra in (('renewal',0.2,0,0),('renewal',0.2,0.3,0),('renewal',0.2,0,0.3),('poisson',1.0,0,0)):
    for T0 in (2,5,10):
        for r in range(100):
            t=renewal(T0,cv,miss,extra)
            if len(t)<10: continue
            g=np.diff(t)/365.25
            Tn=schuster_nfft(t); Tl=lsp_counts(t)
            rows.append(dict(kind=kind,miss=miss,extra=extra,T0=T0,N=len(t),mean=g.mean(),median=np.median(g),
                             nfft=Tn/365.25 if Tn else np.nan,lsp=Tl/365.25))
d=pd.DataFrame(rows)
for c in ['mean','median','nfft','lsp']:
    d['e_'+c]=np.abs(np.log10(d[c]/d.T0)); d['ok_'+c]=d['e_'+c]<=np.log10(1.1)
g=d.groupby(['kind','miss','extra','T0'])
print(g[['e_mean','e_median','e_nfft','e_lsp']].median().round(3))
print(g[['ok_mean','ok_median','ok_nfft','ok_lsp']].mean().round(2))
d.to_csv(f'{OUT}/occ_synthetic.csv',index=False)


                       e_mean  e_median  e_nfft  e_lsp
kind    miss extra T0                                 
poisson 0.0  0.0   2    0.048     0.180   0.245  0.260
                   5    0.067     0.183   0.220  0.225
                   10   0.084     0.205   0.241  0.239
renewal 0.0  0.0   2    0.006     0.009   0.020  0.021
                   5    0.010     0.013   0.025  0.030
                   10   0.014     0.022   0.031  0.035
             0.3   2    0.113     0.072   0.019  0.019
                   5    0.119     0.075   0.029  0.032
                   10   0.119     0.086   0.043  0.045
        0.3  0.0   2    0.150     0.043   0.019  0.024
                   5    0.143     0.036   0.029  0.040
                   10   0.053     0.019   0.036  0.025
                       ok_mean  ok_median  ok_nfft  ok_lsp
kind    miss extra T0                                     
poisson 0.0  0.0   2      0.44       0.04     0.08    0.04
                   5      0.31       0.13     0.10   

In [17]:
RAW={}
def raw(c):
    if c not in RAW:
        d=pd.read_csv(f'{REPO}/data/clustering/clustered_earthquakes_{c}.csv'); d.columns=d.columns.str.lower(); d=d.loc[:,~d.columns.duplicated()]
        d['time']=pd.to_datetime(d['time'],format='mixed',errors='coerce'); RAW[c]=d
    return RAW[c]
def null_series(rt, rm, rng, sd=730.5):
    """inhomogeneous-Poisson null: raw times resampled with Gaussian jitter (sd 2 yr, reflected at the ends), magnitudes permuted, then the 180-day declustering"""
    lo,hi=rt.min(),rt.max()
    t=rng.choice(rt,len(rt))+rng.normal(0,sd,len(rt))
    t=np.where(t<lo,2*lo-t,t); t=np.where(t>hi,2*hi-t,t); t=np.clip(np.round(t),lo,hi).astype(int)
    d,_=fast_decluster(t-t.min(),rng.permutation(rm)); return d
def test(rt,rm,obs_day,rng):
    Tn,Pn,_=schuster_nfft(obs_day,True); Tl,Pl,fap=lsp_counts(obs_day,True)
    nn=[];nl=[]
    for _ in range(B_OCC):
        s=null_series(rt,rm,rng)
        if len(s)<5: nn.append(0); nl.append(0); continue
        nn.append(schuster_nfft(s,True)[1]); nl.append(lsp_counts(s,True)[1])
    pn=(1+np.sum(np.array(nn)>=Pn))/(B_OCC+1); pl=(1+np.sum(np.array(nl)>=Pl))/(B_OCC+1)
    return Tn,pn,Tl,pl,fap
def work(args):
    c,M=args; rng=np.random.default_rng([2026,M,COUNTRIES.index(c)])
    S=SERIES[(c,M)]; d=raw(c); out=[]
    for cid,a in S.items():
        if len(a)<10: continue
        r=d[(d.cluster==cid)&(d.mag>=M)].sort_values('time',kind='mergesort')
        t0=r.time.min()
        rt=((r.time-t0).dt.days).to_numpy(); rm=r.mag.to_numpy()
        day=(a['Day_Number']-a['Day_Number'].min()).to_numpy(float)
        Tn,pn,Tl,pl,fap=test(rt,rm,day,rng)
        row=dict(country=c,cl=cid,M=M,N=len(day),Tn_full=Tn/365.25,pn_full=pn,Tl_full=Tl/365.25,pl_full=pl,fap_full=fap)
        tr,te=splits_stats(a)
        if len(te)>0 and len(tr)>=5:
            cut=pd.to_datetime(tr['time']).max()
            rr=r[r.time<=cut]; rt2=((rr.time-t0).dt.days).to_numpy(); rm2=rr.mag.to_numpy()
            dtr=(pd.to_datetime(tr['time'])-pd.to_datetime(tr['time']).min()).dt.days.to_numpy(float)
            Tn2,pn2,Tl2,pl2,fap2=test(rt2,rm2,np.unique(dtr),rng)
            g=np.diff(np.sort(dtr))
            row.update(Tn_tr=Tn2,pn_tr=pn2,Tl_tr=Tl2,pl_tr=pl2,fap_tr=fap2,med_tr=np.median(g),mean_tr=np.mean(g))
        out.append(row)
    return out

if RECOMPUTE_OCC_NULL:
    OCC=pd.DataFrame([r for c in COUNTRIES for M in range(4,9) for r in work((c,M))]); OCC.to_csv(f'{OUT}/occ_eval_B199.csv',index=False)
else:
    OCC=pd.read_csv(f'{OUT}/occ_eval_B199.csv')
OCC['qn']=bh(OCC.pn_full); OCC['ql']=bh(OCC.pl_full)
print(OCC.groupby('M').agg(n=('N','size'),NFFT_S_p05=('pn_full',lambda s:(s<.05).sum()),NFFT_S_q05=('qn',lambda s:(s<.05).sum()),
      LSP_C_p05=('pl_full',lambda s:(s<.05).sum()),LSP_C_q05=('ql',lambda s:(s<.05).sum()),LSP_C_FAP05=('fap_full',lambda s:(s<.05).sum())))

    n  NFFT_S_p05  NFFT_S_q05  LSP_C_p05  LSP_C_q05  LSP_C_FAP05
M                                                               
4  67           3           0          1          0           12
5  60           4           0          1          0           10
6  40           1           0          3          0            6
7  14           1           0          1          0            0
8   2           0           0          0          0            0


In [18]:
def tobs(tr,te):
    a=pd.to_datetime(tr['time']).max(); b=pd.to_datetime(te['time']).max(); return ((b-a).days/365.25)/len(te)
E=OCC.set_index(['country','cl','M']); res={}; rng=np.random.default_rng(1); BOOT=[]
for M in range(4,8):
    ks=lambda k:{(x[0],x[1]) for x in o.get((k,M),[]) if len(x[2])>0}
    common=sorted(set.intersection(*[ks(k) for k in ['mean','mode','median','NFFT','LSP']]))
    base={(x[0],x[1]):x for x in o[('mean',M)]}
    errs={k:{} for k in ['mean','median','NFFT','LSP','NFFT-S','LSP-C','NFFT-S gated','LSP-C gated']}
    rate={k:[] for k in errs}
    for k in ['mean','median','NFFT','LSP']:
        for x in o[(k,M)]:
            if (x[0],x[1]) in common: errs[k][(x[0],x[1])]=(np.array(x[2],float)/365, x[3])
    for key in common:
        tr,te=base[key][4],base[key][5]; e=E.loc[(key[0],key[1],M)]
        med=e.med_tr
        for k,T in (('NFFT-S',e.Tn_tr),('LSP-C',e.Tl_tr),('NFFT-S gated',e.Tn_tr if e.pn_tr<0.05 else med),('LSP-C gated',e.Tl_tr if e.pl_tr<0.05 else med)):
            errs[k][key]=(np.array(predict_errors(tr,te,T),float)/365, T)
    for k in errs:
        for key,(ee,T) in errs[k].items():
            tr,te=base[key][4],base[key][5]; rate[k].append(np.log10((T/365.25)/tobs(tr,te)))
        ee=np.concatenate([v[0] for v in errs[k].values()])
        s=np.array(rate[k])
        res[(k,M)]=dict(MAE=ee.mean() if ee.size else np.nan,RMSE=np.sqrt((ee**2).mean()) if ee.size else np.nan,MedAE=np.median(ee) if ee.size else np.nan,n=ee.size,
                        medlog=np.median(np.abs(s)),within2=100*np.mean(np.abs(s)<=np.log10(2)),signed=np.median(s),zero=sum(1 for v in errs[k].values() if v[0].size==0))
    TO={c:tobs(base[c][4],base[c][5]) for c in common}
    S={k:np.array([errs[k][c][0].sum() for c in common]) for k in errs}; Nn={k:np.array([errs[k][c][0].size for c in common]) for k in errs}
    L={k:np.array([abs(np.log10((errs[k][c][1]/365.25)/TO[c])) for c in common]) for k in errs}
    for a in ['NFFT-S','LSP-C']:
        for b in ['mean','median']:
            dd=[]; dl=[]
            for _ in range(4000):
                idx=rng.integers(0,len(common),len(common))
                dd.append(S[a][idx].sum()/Nn[a][idx].sum()-S[b][idx].sum()/Nn[b][idx].sum())
                dl.append(np.median(L[a][idx])-np.median(L[b][idx]))
            BOOT.append((M,a,b,*np.percentile(dd,[2.5,97.5]),*np.percentile(dl,[2.5,97.5])))
r=pd.DataFrame(res).T
for v in ['MAE','RMSE','MedAE','n','zero','medlog','within2','signed']:
    print(v); print(r[v].unstack().round(3))
B=pd.DataFrame(BOOT,columns=['M','a','b','mae_lo','mae_hi','rate_lo','rate_hi']); print(B.round(3))
r.to_csv(f'{OUT}/table14_occurrence_oos.csv'); B.to_csv(f'{OUT}/table14_bootstrap.csv',index=False)


MAE
                  4      5      6      7
LSP           0.261  0.406  0.976  1.551
LSP-C         0.259  0.449  0.811  1.378
LSP-C gated   0.266  0.466  0.861  1.376
NFFT          0.299  0.478  0.988  1.291
NFFT-S        0.257  0.448  0.799  1.313
NFFT-S gated  0.267  0.463  0.871  1.385
mean          0.248  0.402  0.801  1.466
median        0.265  0.468  0.871  1.385
RMSE
                  4      5      6      7
LSP           0.345  0.789  2.001  2.580
LSP-C         0.344  0.946  1.714  2.324
LSP-C gated   0.364  0.991  1.745  2.347
NFFT          0.395  0.898  2.067  2.274
NFFT-S        0.343  0.934  1.647  2.218
NFFT-S gated  0.365  0.989  1.759  2.358
mean          0.317  0.838  1.654  2.396
median        0.364  1.000  1.759  2.358
MedAE
                  4      5      6      7
LSP           0.227  0.237  0.296  0.655
LSP-C         0.225  0.248  0.325  0.674
LSP-C gated   0.216  0.249  0.326  0.700
NFFT          0.249  0.260  0.288  0.636
NFFT-S        0.225  0.249  0.341  0.693
N